<a href="https://colab.research.google.com/github/raimund-buehler/visualizing-the-brain/blob/main/notebooks/day_two.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Day Two in Colab"/></a>

# Day Two: Ask Your Own Brain Question

Today your group will run a small fMRI analysis for a question you choose.

You will:

- choose two task conditions to compare,
- write down a prediction,
- fit one first-level model,
- compute your own contrast,
- plot a brain map, and
- explain what the result might mean.

The code is still guided. Your job is to make the scientific choices and interpret the result carefully.

## Researcher mode

Yesterday, we looked at one example: right-hand button presses compared with left-hand button presses.

Today, each group chooses a question. For example:

- **Vision:** Which areas respond more to visual patterns?
- **Audio:** Which areas respond more when the task is heard instead of seen?
- **Language:** Which areas respond more to sentences than to simple visual patterns?
- **Computation challenge:** What changes when people calculate instead of understand sentences?

Before running the analysis, your group should agree on one question and one prediction.

## Setup

Run this cell first. It installs the needed tools in Colab, imports Python packages, loads the fMRI data, and loads the anatomical template.

In [ ]:
# @title Setup: install tools, import packages, and load data
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "nilearn", "nibabel", "matplotlib", "numpy", "pandas"
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import datasets, image, plotting
from nilearn.glm.first_level import FirstLevelModel

%matplotlib inline

anat_img = datasets.load_mni152_template(resolution=2)
fmri_dataset = datasets.fetch_localizer_first_level()
fmri_img = image.load_img(fmri_dataset.epi_img)
events = pd.read_table(fmri_dataset.events)

print("Setup complete.")
print("fMRI image shape:", fmri_img.shape)
print("Number of events:", len(events))

## Choose from these task conditions

The localizer experiment uses these condition names. You can copy them exactly into the next code cell:

```text
horizontal_checkerboard
vertical_checkerboard
sentence_reading
sentence_listening
visual_computation
audio_computation
```

Most groups should start with audio, vision, or language. The computation conditions are challenge options because they can be harder to interpret.

## Choose your research question

Work in your group. Choose one question before editing the code.

Recommended beginner questions:

1. **Audio and language:** `sentence_listening` compared with `sentence_reading`.
2. **Vision:** `horizontal_checkerboard` compared with `vertical_checkerboard`.
3. **Language:** `sentence_reading` compared with `horizontal_checkerboard`.
4. **Language and audio:** `sentence_listening` compared with `horizontal_checkerboard`.
5. **Challenge:** `visual_computation` compared with `sentence_reading`.
6. **Challenge:** `audio_computation` compared with `sentence_listening`.
7. **Harder:** `sentence_listening` compared with `vertical_checkerboard`.

You can also swap Condition A and Condition B. That asks the opposite question and flips what warm and cool colors mean.

Some choices are easier to interpret than others. That is normal in fMRI research. A contrast can show a clear colored cluster, but the reason for that cluster might still be complicated.

The computation choices are challenge options. They may show areas involved in numbers, but they may also show attention, effort, or task difficulty. Do not interpret them as simply "the math area".

Before you continue, discuss these four points as a group:

- Which two conditions are you comparing?
- Which condition do you expect to produce stronger activity?
- Where in the brain do you expect the strongest activity: front/back, left/right, top/bottom?
- Why does your prediction make sense?

## Choose Condition A and Condition B

A contrast compares two conditions:

```text
Condition A - Condition B
```

Warm colors will mean: more response for **Condition A** than **Condition B**.

Cool colors will mean: more response for **Condition B** than **Condition A**.

In the next cell, you can keep the example or replace the condition names inside the quotation marks.

In [ ]:
# Choose two conditions.
# Copy the condition names exactly and keep the quotation marks.

condition_a = "sentence_listening"
condition_b = "sentence_reading"

my_contrast = condition_a + " - " + condition_b

print("Condition A:", condition_a)
print("Condition B:", condition_b)
print("My contrast:", my_contrast)

<details>
<summary>Need ideas? Click for example choices</summary>

```python
# Audio and language: hearing sentences compared with reading sentences
condition_a = "sentence_listening"
condition_b = "sentence_reading"

# Vision: one visual pattern compared with another visual pattern
condition_a = "horizontal_checkerboard"
condition_b = "vertical_checkerboard"

# Language: reading sentences compared with seeing simple visual patterns
condition_a = "sentence_reading"
condition_b = "horizontal_checkerboard"

# Language and audio: hearing sentences compared with seeing a visual pattern
condition_a = "sentence_listening"
condition_b = "horizontal_checkerboard"

# Challenge: doing visual calculations compared with reading sentences
# This may involve numbers, but also attention and task difficulty.
condition_a = "visual_computation"
condition_b = "sentence_reading"

# Challenge: doing calculations from sounds compared with listening to sentences
# This may involve numbers, but also attention and task difficulty.
condition_a = "audio_computation"
condition_b = "sentence_listening"

# Harder: hearing sentences compared with seeing a different visual pattern
# This mixes sound/vision, language, and visual pattern differences, so interpret it carefully.
condition_a = "sentence_listening"
condition_b = "vertical_checkerboard"
```

Choose one pair. Do not run all examples at once.

</details>

## Stop before plotting: make a prediction

Do not rush past this part. Discuss in your group:

1. What will warm colors mean for your contrast?
2. What will cool colors mean?
3. Where do you expect to see the strongest warm colors: front/back, left/right, top/bottom?
4. Are you comparing one clear difference, or do the conditions differ in several ways?

Write one prediction sentence before continuing.

## Fit the model

Now Nilearn checks every voxel. For each voxel, it asks which task timing was useful for explaining that voxel's fMRI signal.

This can take a short moment. You do not need to change this cell.

In [ ]:
first_level_model = FirstLevelModel(
    t_r=2.4,
    slice_time_ref=0,
    hrf_model="spm",
    drift_model="cosine",
    high_pass=0.01,
    smoothing_fwhm=4,
    minimize_memory=True,
)

first_level_model = first_level_model.fit(fmri_img, events=events)
print("Model fitted successfully.")

## Compute your contrast

Now we ask for the comparison you chose above.

In [ ]:
my_map = first_level_model.compute_contrast(
    my_contrast,
    output_type="z_score",
)

print("Contrast calculated:", my_contrast)

## Plot your brain map

The map below shows only stronger differences, using a threshold of `3.0`.

Remember:

- Warm colors mean **Condition A > Condition B**.
- Cool colors mean **Condition B > Condition A**.

You can change the slice heights if your result appears higher or lower in the brain.

In [ ]:
slice_heights = [-10, 10, 30, 50, 70]

plotting.plot_stat_map(
    my_map,
    bg_img=anat_img,
    display_mode="z",
    cut_coords=slice_heights,
    threshold=3.0,
    symmetric_cbar=True,
    title=my_contrast,
)
plotting.show()

### Optional: explore interactively

Run this if you want to click around the result.

In [ ]:
result_viewer = plotting.view_img(
    my_map,
    bg_img=anat_img,
    threshold=3.0,
    title=my_contrast,
)

result_viewer

## Interpret your result

Discuss your result as a group. Use the questions below to organize your interpretation.

### Your contrast

- Which condition was Condition A?
- Which condition was Condition B?
- What did warm colors mean for your contrast?
- What did cool colors mean for your contrast?

### Where you found activity

Use the slice view and the interactive viewer to describe the strongest clusters. You do not need to be perfect. Make your best careful guess.

Quick lobe guide:

- **Occipital lobe:** back of the brain, often important for vision.
- **Temporal lobe:** side/lower part of the brain, often important for hearing and language.
- **Parietal lobe:** top/back part of the brain, often important for attention, space, and number-related tasks.
- **Frontal lobe:** front of the brain, often important for planning, attention, and difficult tasks.

For the strongest warm colors, discuss:

- Are they more on the left, right, or both sides?
- Are they more toward the front, middle, or back?
- Are they more lower, middle, or near the top?
- Which lobe is your best guess?
- Which function might fit that lobe?

For the strongest cool colors, discuss the same questions.

### Your interpretation

Use cautious language. Say what the result might mean, and also name one thing your group is unsure about.

## Be careful

This is real fMRI data, but our analysis is simplified.

- We used one participant.
- We used a simple display threshold.
- Not every colored spot is automatically a discovery.
- Some contrasts are easier to interpret than others.
- A contrast can compare more than one thing at the same time.

For example, `horizontal checkerboard - vertical checkerboard` has a fairly clear idea behind it: the two tasks are very similar, but the visual pattern changes. But `sentence listening - sentence reading` changes more than one thing: hearing versus seeing, spoken language versus written language, and maybe attention. The colored clusters can still be interesting, but we need to be careful about what we claim.

The computation contrasts are especially careful choices. If you compare `visual computation - sentence reading`, a cluster might be related to working with numbers. But it might also be related to the calculation being harder, needing more attention, or using memory. That is why we should say "this might be involved in calculation" instead of "this is the math area".

This is a common problem in fMRI research. A brain map does not explain itself. Scientists have to ask: **What exactly did we compare? What else changed between the two conditions? Could there be another explanation?**

A good scientist can say both what the result suggests and what they are still unsure about.

## Prepare your two-minute group explanation

Each group should be ready to explain:

1. Our research question was...
2. We compared ... minus ...
3. We predicted...
4. Warm colors meant...
5. Cool colors meant...
6. We found...
7. We are still unsure about...

## Sources

- Pinel, P., Thirion, B., Meriaux, S. et al. (2007). Fast reproducible identification and large-scale databasing of individual functional cognitive networks. *BMC Neuroscience, 8*, 91. https://doi.org/10.1186/1471-2202-8-91
- [Nilearn first-level models](https://nilearn.github.io/stable/glm/first_level_model.html)
- [Nilearn plotting tools](https://nilearn.github.io/stable/modules/plotting.html)